In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import gget
from tqdm import tqdm
from scipy.stats import ttest_ind
import matplotlib.colors as mcolors
import os
import networkx as nx
from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

#import rapids_singlecell as rsc
import anndata as an
import scanpy as sc

In [30]:
# print("Working directory set to:", os.getcwd())

Working directory set to: /home/sgolts


## Load Data

In [2]:
# Load TFs
tfpath = "../../resources/allTFs_hg38.txt"

with open(tfpath, 'r') as file:
    tf_list = [line.strip() for line in file]

print(len(tf_list))

# Load adjacencies
control = "/scratch/indikar_root/indikar1/shared_data/HYB/hyb_scenic/control/adj.tsv"
myod1 = "/scratch/indikar_root/indikar1/shared_data/HYB/hyb_scenic/myod1/adj.tsv"
prrx1 = "/scratch/indikar_root/indikar1/shared_data/HYB/hyb_scenic/prrx1/adj.tsv"
prrx1myod1 = "/scratch/indikar_root/indikar1/shared_data/HYB/hyb_scenic/prrx1myod1/adj.tsv"

files = {control, myod1, prrx1, prrx1myod1}

1892


In [3]:
df_list = []
named_dfs = {}  # dictionary to hold each group-specific dataframe

for file in files:
    df = pd.read_csv(file, sep='\t')
    group_name = os.path.basename(os.path.dirname(file))
    df["group"] = group_name
    df_list.append(df)
    named_dfs[f"{group_name}_df"] = df  # save with dynamic name

# Combined dataframe
adf = pd.concat(df_list)
adf['pair'] = adf['TF'] + " → " + adf['target']

print(f"{adf.shape=}")
print(adf.head().to_string())

adf.shape=(16602155, 5)
      TF target  importance  group           pair
0  HMGB2  UBE2C  162.887816  myod1  HMGB2 → UBE2C
1    ID1    ID3  155.020421  myod1      ID1 → ID3
2  HMGB2  TOP2A  154.221592  myod1  HMGB2 → TOP2A
3  HMGB2  CENPF  150.700386  myod1  HMGB2 → CENPF
4  HMGB2  CDCA8  148.412845  myod1  HMGB2 → CDCA8


## Simple Filtering

In [4]:
print(f"Original edges: {len(df)}")

# Filter source
adf = adf[~adf['TF'].str.startswith("MT")]
adf = adf[~adf['TF'].str.startswith("RP")]
print(f"After source filter: {len(adf)} edges")

# Filter target
adf = adf[~adf['target'].str.startswith("MT")]
adf = adf[~adf['target'].str.startswith("RP")]
print(f"After target filter: {len(adf)} edges")

# Reset index
adf = adf.reset_index(drop=True)
print("Index reset.\n")

# Show preview
print(df.head())

Original edges: 4774036
After source filter: 16402768 edges
After target filter: 16052566 edges
Index reset.

      TF  target  importance       group
0  HMGB2   UBE2C  203.719675  prrx1myod1
1  HMGB2   CDCA8  178.349390  prrx1myod1
2   MDM2  CDKN1A  176.899445  prrx1myod1
3  HMGB2   TOP2A  169.720475  prrx1myod1
4  HMGB2   CENPF  164.796976  prrx1myod1


In [5]:
print(adf.shape)
adf.head()

(16052566, 5)


,TF,target,importance,group,pair
0,HMGB2,UBE2C,162.887816,myod1,HMGB2 → UBE2C
1,ID1,ID3,155.020421,myod1,ID1 → ID3
2,HMGB2,TOP2A,154.221592,myod1,HMGB2 → TOP2A
3,HMGB2,CENPF,150.700386,myod1,HMGB2 → CENPF
4,HMGB2,CDCA8,148.412845,myod1,HMGB2 → CDCA8


In [13]:
adf['group'].unique()

array(['myod1', 'prrx1myod1', 'control', 'prrx1'], dtype=object)

In [8]:
# Remove cell cycle and check
# GO cell cycle genes
fpath = "../../resources/human_cell_cycle_genes.csv"
cdf = pd.read_csv(fpath)
go_cc_genes = cdf['gene_name'].unique()
print(f"N unique GO cell cycle genes: {cdf['gene_name'].nunique()}")

# Regev cell cycle genes
fpath = "../../resources/regev_lab_cell_cycle_genes.txt"
regev_genes = [x.strip() for x in open(fpath)]
s_genes = regev_genes[:43]
g2m_genes = regev_genes[43:]


cc_genes = list(set(go_cc_genes) | set(regev_genes))
print(len(cc_genes))

N unique GO cell cycle genes: 142
235


In [11]:
# tmp = adf[~adf['TF'].isin(cc_genes)]
# tmp = tmp[~tmp['target'].isin(cc_genes)]

tmp = adf[~adf['target'].isin(cc_genes)]


tmp.head()

,TF,target,importance,group,pair
1,ID1,ID3,155.020421,myod1,ID1 → ID3
5,HMGB2,PRC1,133.035334,myod1,HMGB2 → PRC1
7,HMGB2,ASPM,123.845665,myod1,HMGB2 → ASPM
10,HMGA2,HMGA2-AS1,121.321675,myod1,HMGA2 → HMGA2-AS1
17,HMGB2,PRR11,100.840485,myod1,HMGB2 → PRR11


In [18]:
tmp[(tmp['group'] == 'prrx1myod1') & (tmp['target'] == 'TGFB1')][['pair', 'importance']].head(20)

,pair,importance
4422049,ETV3 → TGFB1,2.173383
4572272,ZNF33B → TGFB1,1.191069
4585554,ZBTB7A → TGFB1,1.145804
4685831,GADD45A → TGFB1,0.887370
4728952,ID1 → TGFB1,0.808448
4749326,SPZ1 → TGFB1,0.775744
4756681,ZNF549 → TGFB1,0.764553
4827355,ZNF195 → TGFB1,0.671919
4877017,ZNF273 → TGFB1,0.619580
4919426,HMBOX1 → TGFB1,0.580933


In [34]:
for name, df in named_dfs.items():
    print(f"--- Processing {name} ---")
    print(f"Original edges: {len(df)}")

    # Filter source
    df = df[~df['TF'].str.startswith("MT")]
    df = df[~df['TF'].str.startswith("RP")]
    print(f"After source filter: {len(df)} edges")

    # Filter target
    df = df[~df['target'].str.startswith("MT")]
    df = df[~df['target'].str.startswith("RP")]
    print(f"After target filter: {len(df)} edges")

    # Reset index
    df = df.reset_index(drop=True)
    print("Index reset.\n")

    # Update the dictionary with the filtered df
    named_dfs[name] = df

    # Show preview
    print(df.head(), "\n")

--- Processing control_df ---
Original edges: 3962701
After source filter: 3914122 edges
After target filter: 3823820 edges
Index reset.

      TF   target  importance    group
0  HMGB2    UBE2C  224.869547  control
1  HMGB2  ARL6IP1  195.194524  control
2  HMGB1    PTTG1  180.302158  control
3  HMGB1     PTMA  179.470103  control
4  HMGB2     ASPM  172.213782  control 

--- Processing prrx1myod1_df ---
Original edges: 4774036
After source filter: 4717626 edges
After target filter: 4625654 edges
Index reset.

      TF  target  importance       group
0  HMGB2   UBE2C  203.719675  prrx1myod1
1  HMGB2   CDCA8  178.349390  prrx1myod1
2   MDM2  CDKN1A  176.899445  prrx1myod1
3  HMGB2   TOP2A  169.720475  prrx1myod1
4  HMGB2   CENPF  164.796976  prrx1myod1 

--- Processing myod1_df ---
Original edges: 4434074
After source filter: 4381037 edges
After target filter: 4290973 edges
Index reset.

      TF target  importance  group
0  HMGB2  UBE2C  162.887816  myod1
1    ID1    ID3  155.020421  my

In [6]:
# Min-max normalize importance values
threshold = 25
def norm_importance(df):
    """
    Apply min-max normalization and threshold the importance column.
    """
    imp = df['importance']
    imp_norm = (imp - imp.min()) / (imp.max() - imp.min())
    df['importance_norm'] = imp_norm * 100
    df = df[df['importance_norm'] >= threshold].copy()
    return df

In [7]:
adf = norm_importance(adf)
print(adf['importance_norm'].describe())

count    1146.000000
mean       36.908645
std        12.126911
min        25.006499
25%        27.809168
50%        33.137084
75%        41.550264
max       100.000000
Name: importance_norm, dtype: float64


In [23]:
tmp = adf[~adf['target'].isin(cc_genes)]
tmp.head()

,TF,target,importance,group,pair,importance_norm
1,ID1,ID3,155.020421,myod1,ID1 → ID3,66.603091
5,HMGB2,PRC1,133.035334,myod1,HMGB2 → PRC1,57.157401
7,HMGB2,ASPM,123.845665,myod1,HMGB2 → ASPM,53.209145
10,HMGA2,HMGA2-AS1,121.321675,myod1,HMGA2 → HMGA2-AS1,52.124736
17,HMGB2,PRR11,100.840485,myod1,HMGB2 → PRR11,43.325182


In [26]:
# tmp[(tmp['group'] == 'prrx1myod1') & (tmp['target'] == 'TGFB1')][['pair', 'importance_norm']].head(20)
tmp[tmp['group'] == 'myod1'][['pair', 'importance_norm']].head(30)

,pair,importance_norm
1,ID1 → ID3,66.603091
5,HMGB2 → PRC1,57.157401
7,HMGB2 → ASPM,53.209145
10,HMGA2 → HMGA2-AS1,52.124736
17,HMGB2 → PRR11,43.325182
18,ATF3 → PMAIP1,42.281302
20,HMGB2 → CEP55,40.351388
21,HMGB2 → KIF14,40.042826
24,ZNF554 → SETX,39.291392
28,HMGB1 → PRC1,37.537004


In [28]:
tmp[tmp['group'] == 'myod1']['TF'].unique()

array(['ID1', 'HMGB2', 'HMGA2', 'ATF3', 'ZNF554', 'HMGB1', 'ESR1', 'JUN',
       'GADD45A', 'ZNF555', 'DDIT3', 'STAT1', 'RFX1', 'CD59', 'SP110',
       'NFKB1', 'P4HB', 'ID2', 'SP100', 'EGR1', 'RUNX1'], dtype=object)

In [37]:
#df = df[df['importance_norm'] >= 9.001923e-04].copy()

In [38]:
# Apply to each df in named_dfs
for name, df in named_dfs.items():
    print(f"--- Normalizing {name} ---")
    print(f"Before: {len(df)} edges")
    
    df = norm_importance(df)
    named_dfs[name] = df  # update dictionary with normalized/filtered df
    
    print(f"After: {len(df)} edges\n")
    print(df.head(), "\n")

--- Normalizing control_df ---
Before: 3823820 edges
After: 560 edges

      TF   target  importance    group  importance_norm
0  HMGB2    UBE2C  224.869547  control       100.000000
1  HMGB2  ARL6IP1  195.194524  control        86.803450
2  HMGB1    PTTG1  180.302158  control        80.180781
3  HMGB1     PTMA  179.470103  control        79.810764
4  HMGB2     ASPM  172.213782  control        76.583861 

--- Normalizing prrx1myod1_df ---
Before: 4625654 edges
After: 371 edges

      TF  target  importance       group  importance_norm
0  HMGB2   UBE2C  203.719675  prrx1myod1       100.000000
1  HMGB2   CDCA8  178.349390  prrx1myod1        87.546473
2   MDM2  CDKN1A  176.899445  prrx1myod1        86.834737
3  HMGB2   TOP2A  169.720475  prrx1myod1        83.310792
4  HMGB2   CENPF  164.796976  prrx1myod1        80.893991 

--- Normalizing myod1_df ---
Before: 4290973 edges
After: 352 edges

      TF target  importance  group  importance_norm
0  HMGB2  UBE2C  162.887816  myod1       100.0

## Restructure [only for adf]

In [39]:
pdf = adf.copy()

pdf = pd.pivot_table(
    pdf,
    index=['TF', 'target'],
    columns=['group'],
    #values='importance',
    values='importance_norm',
    fill_value=0
)

# Flatten multi-index columns by joining with underscore
pdf.columns = [f"importance_{col}" for col in pdf.columns]
pdf = pdf.reset_index()

print(f"{pdf.shape=}")
print(pdf.columns)
print(pdf[['TF', 'target', 'importance_control','importance_myod1', 'importance_prrx1', 'importance_prrx1myod1']].head().to_string(index=False))

pdf.shape=(677, 6)
Index(['TF', 'target', 'importance_control', 'importance_myod1',
       'importance_prrx1', 'importance_prrx1myod1'],
      dtype='object')
    TF          target  importance_control  importance_myod1  importance_prrx1  importance_prrx1myod1
 AEBP2 ENSG00000205300            0.000000               0.0               0.0              25.126586
 AEBP2 ENSG00000255910            0.000000               0.0               0.0              42.222473
 AEBP2           PATL2            0.000000               0.0               0.0              25.176592
ARID5B           CALD1           25.919844               0.0               0.0               0.000000
  ATF3           BCL10            0.000000               0.0               0.0              26.971832


In [40]:
importance_cols = [col for col in pdf.columns if col.startswith("importance_")]

pdf["n_clusters_with_importance"] = (pdf[importance_cols] > 0).sum(axis=1)

has_multiple = (pdf["n_clusters_with_importance"] > 1).sum()

print("TF-target pairs with importance > 0 in more than one cluster?", has_multiple)

TF-target pairs with importance > 0 in more than one cluster? 249


In [41]:
im_cols = ['importance_control','importance_myod1', 'importance_prrx1', 'importance_prrx1myod1']
fc_columns = []
# Compute log2 fold change of each value vs. mean of other columns in the row --> change to control!!
for col in im_cols:
    others = [c for c in im_cols if c != col]
    new_column = f'log2FC_{col}'
    pdf[new_column] = np.log2((pdf[col] + 1) / (pdf[others].mean(axis=1) + 1))
    fc_columns.append(new_column)

print(pdf[['TF', 'target'] + fc_columns].head().to_string())

       TF           target  log2FC_importance_control  log2FC_importance_myod1  log2FC_importance_prrx1  log2FC_importance_prrx1myod1
0   AEBP2  ENSG00000205300                  -3.228900                -3.228900                -3.228900                      4.707447
1   AEBP2  ENSG00000255910                  -3.914005                -3.914005                -3.914005                      5.433710
2   AEBP2            PATL2                  -3.231463                -3.231463                -3.231463                      4.710205
3  ARID5B            CALD1                   4.750598                -3.269025                -3.269025                     -3.269025
4    ATF3            BCL10                  -3.320573                -3.320573                -3.320573                      4.805903


In [42]:
# Ensure TFs are TFs
pdf = pdf[pdf['TF'].isin(tf_list)]

In [48]:
# TF/TF interactions
tdf = pdf.copy()
tdf = tdf[tdf['TF'].isin(tf_list)]
tdf = tdf[tdf['target'].isin(tf_list)]
print(f"{tdf.shape=}")
tdf.head()

tdf.shape=(49, 11)


,TF,target,importance_control,importance_myod1,importance_prrx1,importance_prrx1myod1,n_clusters_with_importance,log2FC_importance_control,log2FC_importance_myod1,log2FC_importance_prrx1,log2FC_importance_prrx1myod1
7,ATF3,GADD45A,0.000000,0.0,0.0,35.872181,1,-3.695704,-3.695704,-3.695704,5.204461
11,ATF3,IRF1,0.000000,0.0,0.0,29.139202,1,-3.421300,-3.421300,-3.421300,4.913569
12,ATF3,JUN,0.000000,0.0,0.0,27.228718,1,-3.332885,-3.332885,-3.332885,4.819092
26,CANX,HSPA5,38.090606,0.0,0.0,39.522588,2,1.463555,-4.747982,-4.747982,1.564880
38,DDIT3,CEBPG,0.000000,0.0,0.0,28.851749,1,-3.408338,-3.408338,-3.408338,4.899744


## Build Graph

In [44]:
G = nx.from_pandas_edgelist(
    # tdf, # could be pdf for all interactions
    pdf, # could be tdf for tf-tf interactions
    source="TF",
    target="target",
    edge_attr=True,
    create_using=nx.DiGraph(),
)

nx.set_node_attributes(G, {n: True for n in tf_list if n in G}, 'is_tf')

attrs = nx.get_node_attributes(G, 'is_tf')
n_true = sum(attrs.values())
n_total = G.number_of_nodes()

print(f"{n_true} / {n_total} nodes are TFs")

print(G)

68 / 466 nodes are TFs
DiGraph with 466 nodes and 677 edges


In [45]:
# for each

graphs = {}

for name, df in named_dfs.items():
    print(f"--- Building graph for {name} ---")
    
    # Build graph from df
    G = nx.from_pandas_edgelist(
        df,
        source="TF",
        target="target",
        edge_attr=True,
        create_using=nx.DiGraph(),
    )
    
    # Mark TF nodes
    nx.set_node_attributes(G, {n: True for n in tf_list if n in G}, 'is_tf')
    
    # Count TFs
    attrs = nx.get_node_attributes(G, 'is_tf')
    n_true = sum(attrs.values())
    n_total = G.number_of_nodes()
    
    print(f"{n_true} / {n_total} nodes are TFs")
    print(G, "\n")
    
    # Store graph
    graphs[name] = G

--- Building graph for control_df ---
45 / 384 nodes are TFs
DiGraph with 384 nodes and 560 edges 

--- Building graph for prrx1myod1_df ---
48 / 318 nodes are TFs
DiGraph with 318 nodes and 371 edges 

--- Building graph for myod1_df ---
56 / 306 nodes are TFs
DiGraph with 306 nodes and 352 edges 

--- Building graph for prrx1_df ---
23 / 212 nodes are TFs
DiGraph with 212 nodes and 253 edges 

